# Error Analysis (M1)\n
This notebook shows one concrete failure mode and one fix attempt.

In [ ]:
from pathlib import Path\n
import json\n
import sys\n
root = Path('..').resolve()\n
if str(root) not in sys.path:\n
    sys.path.insert(0, str(root))\n
from src.semantic_retriever import SemanticRetriever, build_index_from_corpus

In [ ]:
corpus = [json.loads(x) for x in (root / 'data' / 'corpus.jsonl').read_text(encoding='utf-8').splitlines() if x.strip()]\n
chunk_ids, chunk_texts = build_index_from_corpus(corpus)

In [ ]:
query = 'rule for x cubed derivative'\n
baseline = SemanticRetriever(ngram_range=(1,1), max_features=2000, min_df=2)\n
baseline.fit(chunk_texts, chunk_ids)\n
baseline_hits = baseline.query(query, top_k=3)\n
[(h.doc_id, round(h.score, 4)) for h in baseline_hits]

Concrete failure mode: baseline uses `min_df=2`, so rare but important terms (like 'cubed') are dropped from vocabulary.

In [ ]:
improved = SemanticRetriever(ngram_range=(1,2), max_features=2000, min_df=1)\n
improved.fit(chunk_texts, chunk_ids)\n
improved_hits = improved.query(query, top_k=3)\n
[(h.doc_id, round(h.score, 4)) for h in improved_hits]

Fix attempt: keep rare terms (`min_df=1`) and enable bi-grams (`ngram_range=(1,2)`) so phrase-level math cues are kept.